In [ ]:
import pandas as pd
from qdrant_client import QdrantClient

# 连接到你的远程服务器（通过你刚才做的 VS Code 端口转发）
client = QdrantClient("localhost", port=6333)
# 打印所有的集合名称
print(client.get_collections())

In [ ]:
!pip install tabulate

In [ ]:
import pandas as pd
from qdrant_client import QdrantClient

# 连接到你的远程服务器（通过你刚才做的 VS Code 端口转发）
client = QdrantClient("localhost", port=6333)

COLLECTION_NAME = "libero_goal_task_362"  # 记得改成你实际建的表名

try:
    # 1. 获取集合信息（看看有多少条数据）
    info = client.get_collection(COLLECTION_NAME)
    print(f"✅ 集合状态: {info.status}")
    print(f"📊 数据总量: {info.points_count} 条")
    
    if info.points_count > 0:
        # 2. 拉取前 10 条数据看看长什么样
        res = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=10,
            with_payload=True,
            with_vectors=False # 向量是一堆数字，肉眼看不懂，就不拉了
        )[0]
        
        # 3. 用 Pandas 打印漂亮的表格
        print("\n👀 前 10 条数据预览：")
        df = pd.DataFrame([point.payload for point in res])
        # 加上 ID 列
        df.insert(0, 'id', [point.id for point in res])
        print(df.to_markdown(index=False)) # 或者直接 print(df)
    else:
        print("⚠️ 集合是空的，快去灌数据吧！")

except Exception as e:
    print(f"❌ 出错了: {e}")
    print("提示：确认 Collection 名字对不对？确认端口转发开启了吗？")

In [ ]:
# 只能使用gpu1
import os
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
# 指向具体的数据集版本目录，例如 libero_spatial_no_noops/1.0.0
db_dir = '/path/to/rtcache/libero/datasets--openvla--modified_libero_rlds/snapshots/6ce6aaaaabdbe590b1eef5cd29c0d33f14a08551/libero_goal_no_noops/1.0.0'
builder = tfds.builder_from_directory(db_dir)

In [ ]:
# 2. 获取数据集的元数据 (Metadata)
print("=== 数据集信息 ===")
print(f"数据集名称: {builder.name}")
print(f"版本: {builder.version}")
print(f"包含的 Split (划分): {builder.info.splits.keys()}")
# print(f"总样本数 (Test): {builder.info.splits['test'].num_examples}")
print(f"总样本数 (Train): {builder.info.splits['train'].num_examples}")
print("-" * 30)

In [ ]:
# 加载一部分数据示例
# 方式 1: 加载前 10 个样本 (用于快速测试)
ds = builder.as_dataset(split='train[:10]')

# 方式 2: 加载前 10% 的样本
# ds = builder.as_dataset(split='train[:10%]')

# 方式 3: 加载第 10 到第 20 个样本
# ds = builder.as_dataset(split='train[10:20]')

# 方式 4: 加载全部 (默认)
# ds = builder.as_dataset(split='train')

In [ ]:
for episode in ds.take(1):
    print("\n=== Episode (轨迹) 结构 ===")
    print("Episode Keys:", episode.keys())
    # 常见的 Episode key 有: 'steps', 'episode_metadata' (如 task_id, file_path)
    
    if 'episode_metadata' in episode:
            print("Episode Metadata:", episode['episode_metadata'])

    # 5. 查看轨迹中的具体步骤 (Steps)
    # 'steps' 是一个嵌套的 Dataset，包含了每一步的 (Obs, Action, Reward)
    steps = episode['steps']
    
    print("\n=== Step (单步) 内容示例 ===")
    # 取出这一条轨迹中的第一帧/第一步
    for step in steps.take(1):
        # 递归打印字典结构和 Tensor 形状
        for key, value in step.items():
            if isinstance(value, dict):
                print(f"Key: '{key}'")
                for sub_key, sub_val in value.items():
                    print(f"  - {sub_key}: shape={sub_val.shape}, dtype={sub_val.dtype}")
            else:
                print(f"Key: '{key}': shape={value.shape}, dtype={value.dtype}")
                
        # 如果你想看具体的 Action 值
        print("\n示例 Action 数据:", step['action'].numpy())
        # 如果你想看具体的 Reward
        print("示例 Reward 数据:", step['reward'].numpy())

In [ ]:
import matplotlib.pyplot as plt

# 显示第一个 episode 的第一步的图像
for episode in ds.take(1):
    steps = episode['steps']
    
    for step in steps.take(1):
        # 获取图像数据
        image_data = step['observation']['image']
        
        # 显示图像
        plt.figure(figsize=(8, 6))
        plt.imshow(image_data)
        plt.title("Observation Image")
        plt.axis('off')
        plt.show()
        
        print(f"图像形状: {image_data.shape}")
        print(f"图像数据类型: {image_data.dtype}")
        # print(f"像素值范围: [{image_data.min()}, {image_data.max()}]")

In [ ]:
# 插入一条轨迹到数据库 (修复版)
import os
import sys
import time
import uuid
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from PIL import Image
from io import BytesIO
import requests
from pymongo import MongoClient
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams
from pathlib import Path
import base64
import torch

# 确保使用 GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# 配置
MONGO_URL = "mongodb://localhost:27017/"
MONGO_DB_NAME = "OpenVLACollection"
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
EMBEDDING_SERVER_URL = "http://127.0.0.1:9020/predict"
IMAGE_STORAGE_PATH = "./data/images"

# 向量维度配置 (需要与 Embedding Server 输出一致)
OPENVLA_DIM = 2176
CLIP_DIM = 512

def insert_single_trajectory():
    # 1. 连接数据库
    print("Connecting to databases...")
    mongo_client = MongoClient(MONGO_URL)
    mongo_db = mongo_client[MONGO_DB_NAME]
    mongo_collection = mongo_db["trajectories"]
    
    qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
    
    # --- 修复 3: 确保 Qdrant Collection 存在 ---
    collections = [
        ("image_collection", OPENVLA_DIM),
        ("clip_image_collection", CLIP_DIM)
    ]
    
    for name, dim in collections:
        if not qdrant_client.collection_exists(name):
            print(f"Creating collection {name} with dim={dim}...")
            qdrant_client.create_collection(
                collection_name=name,
                vectors_config=VectorParams(size=dim, distance="Cosine")
            )
        else:
            print(f"Collection {name} already exists.")
    
    # 2. 加载数据集 (使用你之前定义的 ds)
    # 假设 ds 已经在前面的单元格中加载好了，如果没有，请取消下面几行的注释
    # db_dir = '/path/to/rtcache/libero/datasets--openvla--modified_libero_rlds/snapshots/6ce6aaaaabdbe590b1eef5cd29c0d33f14a08551/libero_goal_no_noops/1.0.0'
    # builder = tfds.builder_from_directory(db_dir)
    # ds = builder.as_dataset(split='train[:1]')
    
    dataset_name = "libero_goal_no_noops"
    
    # 3. 处理并插入
    for episode_idx, episode in enumerate(ds.take(1)):
        print(f"Processing episode {episode_idx}...")
        steps = list(episode['steps'].as_numpy_iterator())
        total_steps = len(steps)
        
        # --- 修复 1: 正确提取文本 ---
        # 文本通常在 Step 的顶层，而不是 observation 里面
        # 我们取第一步的 language_instruction 作为整个 episode 的文本
        first_step = steps[0]
        if 'language_instruction' in first_step:
            episode_text = first_step['language_instruction'].decode('utf-8')
        else:
            episode_text = ""
        print(f"  Instruction: {episode_text}")
        
        batch_points_image = []
        batch_points_clip = []
        mongo_docs = []
        
        for step_idx, step in enumerate(steps):
            # --- 修复 2: 正确提取 Action ---
            # Libero 的 action 直接就是 (7,) 的 numpy array
            raw_action = step['action']
            
            # 简单的归一化 (示例逻辑，根据需要调整)
            normalized_action = raw_action.copy()
            normalized_action[:3] = np.clip(normalized_action[:3], -0.1, 0.1)
            normalized_action[3:6] = np.clip(normalized_action[3:6], -0.5, 0.5)
            normalized_action[6] = 1.0 if normalized_action[6] > 0 else 0.0
            
            # --- 提取图像 ---
            image_data = step['observation']['image']
            image = Image.fromarray(image_data)
            
            # 保存图像
            doc_id = f"{dataset_name}_test_{episode_idx}_{step_idx}"
            image_dir = Path(IMAGE_STORAGE_PATH)
            image_dir.mkdir(parents=True, exist_ok=True)
            image_path = image_dir / f"{doc_id}.png"
            image.save(image_path)
            
            # --- 生成 Embedding ---
            buf = BytesIO()
            image.save(buf, format='PNG')
            buf.seek(0)
            files = {"file": ("image.png", buf, "image/png")}
            data = {"instruction": episode_text, "option": "image"} 
            
            try:
                resp = requests.post(EMBEDDING_SERVER_URL, files=files, data=data, timeout=30)
                resp.raise_for_status()
                emb_result = resp.json()
            except Exception as e:
                print(f"  Error getting embedding for step {step_idx}: {e}")
                continue

            # --- 准备数据库记录 ---
            
            # MongoDB 文档
            mongo_doc = {
                'id': doc_id,
                'dataset_name': dataset_name,
                'episode_idx': episode_idx,
                'step_idx': step_idx,
                'total_steps': total_steps,
                'raw_action': raw_action.tolist(), # 存入 list
                'normalized_action': normalized_action.tolist(),
                'text': episode_text, # 存入正确的文本
                'image_path': str(image_path)
            }
            mongo_docs.append(mongo_doc)
            
            # Qdrant Points
            if "image_features" in emb_result:
                b64 = emb_result["image_features"]
                tensor = torch.load(BytesIO(base64.b64decode(b64)), map_location="cpu")
                vector = tensor.squeeze(0).tolist()
                
                point = PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload={
                        'logical_id': doc_id,
                        'dataset_name': dataset_name,
                        'episode_idx': episode_idx,
                        'step_idx': step_idx,
                        'text': episode_text
                    }
                )
                batch_points_image.append(point)

            if "clip_image_features" in emb_result:
                b64 = emb_result["clip_image_features"]
                tensor = torch.load(BytesIO(base64.b64decode(b64)), map_location="cpu")
                vector = tensor.squeeze(0).tolist()
                
                point = PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload={
                        'logical_id': doc_id,
                        'dataset_name': dataset_name,
                        'episode_idx': episode_idx,
                        'step_idx': step_idx,
                        'text': episode_text
                    }
                )
                batch_points_clip.append(point)
                
            if step_idx % 10 == 0:
                print(f"  Processed step {step_idx}/{total_steps}")

        # --- 批量插入 ---
        if mongo_docs:
            print(f"Inserting {len(mongo_docs)} documents to MongoDB...")
            mongo_collection.insert_many(mongo_docs)
            
        if batch_points_image:
            print(f"Inserting {len(batch_points_image)} points to Qdrant (image_collection)...")
            qdrant_client.upsert(
                collection_name="image_collection",
                points=batch_points_image
            )
            
        if batch_points_clip:
            print(f"Inserting {len(batch_points_clip)} points to Qdrant (clip_image_collection)...")
            qdrant_client.upsert(
                collection_name="clip_image_collection",
                points=batch_points_clip
            )
            
    print("Done! Trajectory inserted with correct Action and Text.")

# 运行插入函数
insert_single_trajectory()

In [ ]:
# 单元格 1: 直接向量数据库检索测试
import time
import random
import requests
import torch
import numpy as np
from PIL import Image
from io import BytesIO
from pymongo import MongoClient
from qdrant_client import QdrantClient
import base64

# 配置
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
MONGO_URL = "mongodb://localhost:27017/"
MONGO_DB_NAME = "OpenVLACollection"
EMBEDDING_SERVER_URL = "http://127.0.0.1:9020/predict"
COLLECTION_NAME = "image_collection" 

def test_direct_db_retrieval():
    print("=== 直接数据库检索测试 (n=1) ===")
    
    # 1. 随机采样
    # 假设 ds 已经加载 (如果未加载，请取消下面注释)
    # builder = tfds.builder_from_directory(db_dir)
    # ds = builder.as_dataset(split='train[:100]') 
    
    # 从前 10 个 Episode 中随机选一个
    episode = next(iter(ds.shuffle(10).take(1)))
    steps = list(episode['steps'].as_numpy_iterator())
    
    # 随机选一步
    step_idx = random.randint(0, len(steps)-1)
    step = steps[step_idx]
    
    # 获取数据
    image_data = step['observation']['image']
    image = Image.fromarray(image_data)
    gt_action = step['action']
    
    if 'language_instruction' in steps[0]:
        instruction = steps[0]['language_instruction'].decode('utf-8')
    else:
        instruction = ""
        
    print(f"Selected: Episode ?, Step {step_idx}")
    print(f"Instruction: {instruction}")
    print(f"GT Action: {gt_action}")
    
    # 2. 生成 Embedding
    t0 = time.time()
    buf = BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    files = {"file": ("image.png", buf, "image/png")}
    data = {"instruction": instruction, "option": "image"}
    
    resp = requests.post(EMBEDDING_SERVER_URL, files=files, data=data)
    emb_result = resp.json()
    
    if COLLECTION_NAME == "image_collection":
        b64_feat = emb_result["image_features"]
    else:
        b64_feat = emb_result["clip_image_features"]
        
    tensor = torch.load(BytesIO(base64.b64decode(b64_feat)), map_location="cpu")
    query_vector = tensor.squeeze(0).tolist()
    t1 = time.time()
    print(f"Embedding Time: {(t1-t0)*1000:.2f} ms")
    
    # 3. Qdrant 查询
    client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
    t2 = time.time()
    if hasattr(client, 'search'):
        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            limit=1,
            with_payload=True
        )
    else:
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=1,
            with_payload=True
        ).points
    t3 = time.time()
    print(f"Vector DB Query Time: {(t3-t2)*1000:.2f} ms")
    
    if not search_result:
        print("No results found in Qdrant.")
        return

    top1 = search_result[0]
    logical_id = top1.payload.get('logical_id')
    print(f"Top 1 Score: {top1.score:.4f}")
    print(f"Top 1 ID: {logical_id}")
    
    # 4. MongoDB 查询
    t4 = time.time()
    mongo_client = MongoClient(MONGO_URL)
    mongo_coll = mongo_client[MONGO_DB_NAME]["trajectories"]
    
    # 注意：数据库中的 id 字段存储的是 logical_id 字符串
    doc = mongo_coll.find_one({"id": logical_id})
    t5 = time.time()
    print(f"MongoDB Query Time: {(t5-t4)*1000:.2f} ms")
    
    if doc:
        retrieved_action = np.array(doc['raw_action'])
        print(f"Retrieved Action (Top 1): {retrieved_action}")
        
        mse = np.mean((gt_action - retrieved_action)**2)
        print(f"MSE (GT vs Retrieved): {mse:.6f}")
    else:
        print("Document not found in MongoDB.")

test_direct_db_retrieval()

In [ ]:
# 单元格 2: API 检索测试
import time
import random
import requests
import numpy as np
from PIL import Image
from io import BytesIO

# 配置
RETRIEVAL_URL = "http://127.0.0.1:5002/pipeline"

def test_api_retrieval():
    print("\n=== API 检索测试 (n=1) ===")
    
    # 1. 随机采样 (复用 ds)
    episode = next(iter(ds.shuffle(10).take(1)))
    steps = list(episode['steps'].as_numpy_iterator())
    step_idx = random.randint(0, len(steps)-1)
    step = steps[step_idx]
    
    image_data = step['observation']['image']
    image = Image.fromarray(image_data)
    gt_action = step['action']
    
    if 'language_instruction' in steps[0]:
        instruction = steps[0]['language_instruction'].decode('utf-8')
    else:
        instruction = ""
        
    print(f"Selected: Episode ?, Step {step_idx}")
    print(f"Instruction: {instruction}")
    print(f"GT Action: {gt_action}")
    
    # 2. 调用 API
    buf = BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    
    files = {"file": ("image.png", buf, "image/png")}
    data = {
        "instruction": instruction,
        "option": "both"
    }
    
    t0 = time.time()
    try:
        response = requests.post(RETRIEVAL_URL, files=files, data=data)
        t1 = time.time()
        latency = (t1 - t0) * 1000
        print(f"API Total Time: {latency:.2f} ms")
        
        if response.status_code == 200:
            result = response.json()
            print(f"API Result: {result}") # 调试用
            
            # 提取 Action
            pred_action = None
            
            # 检查 rtcache_trajectory
            if 'rtcache_trajectory' in result and result['rtcache_trajectory']:
                traj = np.array(result['rtcache_trajectory'])
                print(f"Retrieved Trajectory Shape: {traj.shape}")
                print(f"Full Retrieved Trajectory (All Steps):\n{traj}")
                
                # 如果是列表的列表，取第一个
                if traj.ndim > 1:
                    pred_action = traj[0]
                    print(f"Using first step of trajectory for MSE calculation.")
                else:
                    pred_action = traj
            # 检查 averaged_trajectory
            elif 'averaged_trajectory' in result and result['averaged_trajectory']:
                traj = np.array(result['averaged_trajectory'])
                print(f"Retrieved Trajectory Shape: {traj.shape}")
                print(f"Full Retrieved Trajectory (All Steps):\n{traj}")
                
                if traj.ndim > 1:
                    pred_action = traj[0]
                    print(f"Using first step of trajectory for MSE calculation.")
                else:
                    pred_action = traj
            
            if pred_action is not None:
                print(f"Retrieved Action (API, Step 0): {pred_action}")
                mse = np.mean((gt_action - pred_action)**2)
                print(f"MSE (GT vs API Step 0): {mse:.6f}")
            else:
                print("No action found in API response.")
                print(f"Response keys: {result.keys()}")
                
        else:
            print(f"API Failed: {response.status_code} - {response.text}")
            
    except Exception as e:
        print(f"API Error: {e}")

test_api_retrieval()